In [1]:
from common.neo4j_client import Neo4jCustomClient
from common.llm_client import LLMClient
from common.embedding_client import EmbeddingClient

neo4j_client = Neo4jCustomClient()
llm = LLMClient()
embedding_client = EmbeddingClient()

bolt://localhost:7687


In [2]:
print(f"Neo4j client: {neo4j_client.verify_connectivity()}")
print(f"LLM Model{llm.get_model_name()}")
print(
    f"Embedding Model {embedding_client.get_model_name()} (Size: {embedding_client.get_embedding_dimension('test')})"
)

Neo4j client: True
LLM Modelgoogle/gemma-4-e4b
Embedding Model text-embedding-bge-m3 (Size: 1024)


In [5]:
schema_string = neo4j_client.get_schema()

In [6]:
print(schema_string)

Node properties:
Chunk {embedding: LIST, index: INTEGER, text: STRING}
PDF {id: STRING}
Parent {id: STRING, text: STRING}
Child {id: STRING, embedding: LIST, text: STRING}
Relationship properties:

The relationships:
(:PDF)-[:HAS_PARENT]->(:Parent)
(:Parent)-[:HAS_CHILD]->(:Child)


In [3]:
query = "match-all"
print(query)
# neo4j_client.visualize_graph(query)
html_path = neo4j_client.save_graph_png(query)
print(html_path)

match-all
<coroutine object Neo4jCustomClient.save_graph_png at 0x12f2a87b0>


In [2]:
from IPython.display import Image, display
html_path = neo4j_client.save_graph_html("match-all")
png_path = await neo4j_client.save_graph_png(html_path)

print(type(html_path), html_path)
print(type(png_path), png_path)

#display(Image(filename=str(png_path)))

<class 'pathlib.PosixPath'> /Users/jbd/ai/graph-rag/03/image/match-all.html
<class 'pathlib.PosixPath'> /Users/jbd/ai/graph-rag/03/image/match-all.png


In [ ]:
png_path = await neo4j_client.save_graph_png(html_path)
print(png_path)

In [ ]:
display(Image(filename=str(png_path)))

In [11]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    """
You are an expert in generating Cypher queries.

Schema:
{schema}

Terminology:
{terminology}

Examples:
{examples}

Question:
{question}

Return only the Cypher query.
"""
)

In [12]:
question = "저장된 문서 청크는 총 몇 개인가요?"

terminology_string = """
- 문서 청크(document chunk)는 Chunk 노드로 표현됩니다.
- 청크의 순서는 index 속성에 저장됩니다.
- 청크의 본문은 text 속성에 저장됩니다.
- 청크의 임베딩 벡터는 embedding 속성에 저장됩니다.
"""

examples = [
    [
        "첫 번째 문서 청크를 보여주세요.",
        """
        MATCH (c:Chunk)
        RETURN c.index, c.text
        ORDER BY c.index
        LIMIT 1
        """
    ]
]

formatted_examples = "\n".join(
    f"Question: {example_question}\nCypher: {cypher}"
    for example_question, cypher in examples
)

full_prompt = prompt_template.format(
    question=question,
    schema=schema_string,
    terminology=terminology_string,
    examples=formatted_examples,
)

print(full_prompt)


You are an expert in generating Cypher queries.

Schema:
Node properties:
Chunk {embedding: LIST, index: INTEGER, text: STRING}
PDF {id: STRING}
Parent {id: STRING, text: STRING}
Child {id: STRING, embedding: LIST, text: STRING}
Relationship properties:

The relationships:
(:PDF)-[:HAS_PARENT]->(:Parent)
(:Parent)-[:HAS_CHILD]->(:Child)

Terminology:

- 문서 청크(document chunk)는 Chunk 노드로 표현됩니다.
- 청크의 순서는 index 속성에 저장됩니다.
- 청크의 본문은 text 속성에 저장됩니다.
- 청크의 임베딩 벡터는 embedding 속성에 저장됩니다.


Examples:
Question: 첫 번째 문서 청크를 보여주세요.
Cypher: 
        MATCH (c:Chunk)
        RETURN c.index, c.text
        ORDER BY c.index
        LIMIT 1
        

Question:
저장된 문서 청크는 총 몇 개인가요?

Return only the Cypher query.



In [13]:
response = llm.invoke(full_prompt)

print(response)

MATCH (c:Chunk)
RETURN count(c)


In [14]:
result = neo4j_client.driver.execute_query(response)

In [15]:
print(result)

EagerResult(records=[<Record count(c)=298>], summary=<neo4j._work.summary.ResultSummary object at 0x13424b350>, keys=['count(c)'])
